## Baseline (no fine-tuning) — Task 1 (Risk Clause Recognition), Hard Negatives

This notebook is an **exact replica** of [llm_fine_tuning_LORA_task1_v2.ipynb](llm_fine_tuning_LORA_task1_v2.ipynb) with **one difference: there is no training**. The base **`meta-llama/Meta-Llama-3.1-8B`** model is loaded and used *as-is* to make predictions on the **same validation set** the fine-tuned run is evaluated on, so the two sets of metrics are directly comparable.

Task 1 = given a single contract clause excerpt, decide whether it is an instance of a specific risk clause category (`Yes`) or not (`No`) across the 32 Yes/No categories.

**Why this baseline matters:** the fine-tuning notebook applies a *hard negatives* fix (every `No` example borrows a real clause from a different category in the same contract) so both `Yes` and `No` inputs are genuine legal text and the model must actually recognize the clause type. Running the **un-fine-tuned** base model on the identical validation set tells us what the model can do out-of-the-box — the reference point that fine-tuning is measured against.

The data pipeline (load → hard negatives → contract-level split → balance → save JSONL) is byte-for-byte identical to the fine-tuning notebook (same `random.seed(42)`), which guarantees the validation set here is exactly the one used there. All artifacts are written to a **separate `no_finetune_baseline/` subdirectory** so they never overwrite the fine-tuned run's outputs.


In [ ]:
# --- Pin to a single GPU BEFORE torch is imported anywhere ---
# Kaggle "GPU T4 x2" exposes 2 GPUs. device_map="auto" then shards the model
# across cuda:0/cuda:1. At loss time TRL's _chunked_cross_entropy_loss builds the
# label mask on cuda:0 while the final hidden_states/lm_head live on cuda:1 ->
# "indices should be either on cpu or on the same device as the indexed tensor
# (cuda:1)". An 8B model in 4-bit (~5-6 GB) fits in ONE T4 (16 GB), so hide GPU 1.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Kaggle: install the library versions this notebook expects (no-op locally) ---
# The Kaggle base image ships older trl/peft; pin trl 1.x so SFTConfig,
# completion_only_loss and processing_class are available.
if os.path.exists("/kaggle"):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.55.4",   # MUST be <4.56: the 4.56 "core_model_loading"
                                              # threaded loader breaks bnb 4-bit -> full fp16 load -> OOM
                    "bitsandbytes==0.46.1",
                    "accelerate==1.7.0",
                    "peft==0.15.2",
                    "trl==0.20.0",
                    "datasets"], check=True)



In [ ]:
import sys; print("UTF-8 mode:", sys.flags.utf8_mode)

# Step 1 : Load data (sampled master_clauses file from CUAD)

Dataset Description Summarized : 

1. Columns NOT ending in "Answer" (Context Columns)
- Role: These columns contain the text context (the actual excerpt or "clause") extracted from the contract.
- Content: A string of text directly from the contract that is responsive to a specific category.
- Purpose: This serves as the "evidence" or the "source passage" that justifies a specific determination.
- Handling of Omissions: If parts of the text are irrelevant, they may be replaced with <omitted>.

2. Columns ending in "Answer" (Label Columns)
- Role: These columns contain the derived human-input answers based on the text context found in the corresponding Context column.
- Content:
- For "Yes/No" Categories (32 types): The value is "Yes" if the clause exists, or "No" if no string was found. (e.g., Termination for Convenience).
- For Extraction Categories (Task 2): The value is a normalized string representing a specific entity, date, or number.
- Purpose: This is the "ground truth" or "label" for the machine learning task.

In [ ]:
import pandas as pd
import json
from pathlib import Path
import csv
import re
from sklearn.model_selection import train_test_split

In [ ]:
# --- Environment config: run unchanged locally OR on Kaggle ---
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()

if ON_KAGGLE:
    # Read-only mounted dataset (matches dataset-metadata.json id slug + folder)
    DATA_DIR = Path("/kaggle/input/cuad-master-clauses-cleaned")
    # Only this dir is writable AND persisted as kernel output:
    WORK_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path(os.getenv("DATA_DIR", "data")) / "CUAD_v1"
    WORK_DIR = Path(".")

print(f"ON_KAGGLE={ON_KAGGLE} | DATA_DIR={DATA_DIR} | WORK_DIR={WORK_DIR}")

# All artifacts from THIS (no-fine-tune) notebook go under a dedicated subdirectory
# so they never overwrite the fine-tuned run's outputs (eval_metrics.json, cuad/, ...).
OUTPUT_DIR = WORK_DIR / "no_finetune_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"OUTPUT_DIR={OUTPUT_DIR}")


In [ ]:
CUAD_PATH = DATA_DIR   # set by the environment-config cell
# Step 1: load the cleaned CSV.
#CSV_NAME = 'master_clauses_cleaned_sampled.csv'   ~100 records, quick iteration
CSV_NAME = 'master_clauses_cleaned.csv'        # full dataset

# Robustly locate the CSV. On Kaggle the dataset may mount under a different folder
# than expected (or not be attached at all) - search /kaggle/input as a fallback.
MASTER_CLAUSES_PATH = CUAD_PATH / CSV_NAME
if not MASTER_CLAUSES_PATH.exists():
    search_root = Path('/kaggle/input') if ON_KAGGLE else CUAD_PATH
    matches = list(search_root.rglob(CSV_NAME)) if search_root.exists() else []
    if matches:
        MASTER_CLAUSES_PATH = matches[0]
        print(f'Resolved CSV via fallback search: {MASTER_CLAUSES_PATH}')
    else:
        print(f'Could NOT find {CSV_NAME} under {search_root}. Actually mounted:')
        listing = sorted(search_root.rglob('*'))[:50] if search_root.exists() else []
        for _p in listing:
            print('   ', _p)
        if not listing:
            print('   (nothing — the dataset is not attached to this kernel)')
        raise FileNotFoundError(f'{CSV_NAME} not found under {search_root}')

# Read the file manually using the CSV module to handle inconsistencies
data = []
with open(MASTER_CLAUSES_PATH, 'r', encoding='utf-8', errors='replace') as f:
    reader = csv.DictReader(f)
    for row in reader:
        data.append(row)

df = pd.DataFrame(data)
print(f'Data Loaded Successfully from {MASTER_CLAUSES_PATH}')
print(f'Total Contracts: {len(df)}')
print(df.head(3))

In [ ]:
df.head(3)

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove special characters but keep spaces
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

# Clean column names
df.columns = [clean_text(col).strip() for col in df.columns]
for col in df.columns:
    print(col)

In [ ]:
new_columns = {}
for col in df.columns:
    print(f"Processing column: '{col}'")
    if "Answer" in col:
            # Remove "Answer" from the string and append "_Answer" at the end
            new_columns[col] = f"{col.replace('Answer', '').strip()}_Answer"

df = df.rename(columns=new_columns)

In [ ]:
for col in df.columns:
    print(col)

# Step 2 (cont.) : Restrict to the 32 Task 1 categories

Per the plan, the load / column-clean / `_Answer`-rename cells above are left unchanged. Here we derive the Task 1 category list by excluding the Task 2 entity-extraction fields, so Task 2 fields never enter Task 1 training.

In [ ]:
task2_categories = [
    "Filename", "Document Name", "Parties", "Agreement Date", "Effective Date",
    "Expiration Date", "Renewal Term", "Notice Period To Terminate Renewal",
    "Governing Law", "Warranty Duration",
]
task1_categories = [
    col for col in df.columns
    if not col.endswith("_Answer") and col.strip() != ""
    and col not in task2_categories
    and f"{col}_Answer" in df.columns
]
assert len(task1_categories) == 32, f"Expected 32 Task 1 categories, got {len(task1_categories)}"
print(f"{len(task1_categories)} Task 1 categories:")
for c in task1_categories:
    print(" -", c)

In [ ]:
def save_jsonl(data, filename):
    with open(filename, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry) + '\n')

# Step 3 & 4 : Build binary (Yes/No) examples with **hard negatives**, split by contract

For each contract row we gather **all clauses actually present** in that contract, keyed by category. Then for each of the 32 categories we build one example:

- **Yes** → use that category's own real clause text.
- **No** → randomly borrow a real clause from a **different** present category (a *hard negative*). If the contract has no other clause to borrow, skip the example.

So both `Yes` and `No` inputs are genuine legal text — the model can no longer cheat off a placeholder, and the only way to answer is to recognize the clause type. The instruction wording is *"Is the following contract text a `...` clause?"* because we now feed a single clause excerpt.

The split is done on **contracts (df rows) first** (Step 4), then examples are built from each side, to prevent a contract leaking across train/val.

In [ ]:
import random
random.seed(42)  # fix randomness so the same hard negatives are picked every run

# Turn a raw answer cell into a clean Yes/No label.
# "No" or blank -> "No"; anything else (real clause text) -> "Yes".
def to_binary(answer):
    return "No" if (pd.isna(answer) or str(answer).strip().lower() == "no") else "Yes"

# Return the clause text in a column, or None if that cell is empty/blank.
def nonempty_context(row, cat):
    v = row.get(cat)
    return str(v).strip() if pd.notna(v) and str(v).strip() else None

# GOAL OF THIS FUNCTION:
# Turn a table of contracts into individual training examples for Task 1.
# Each example is one yes/no question: "is THIS piece of text an example of
# category X?". The key trick (hard negatives) is that the "No" examples are NOT
# a placeholder — they are real clause text taken from a DIFFERENT category in the
# same contract, so the model has to actually understand the clause to answer.
def build_examples(frame):
    rows = []  # all finished examples, one dict per question
    for _, row in frame.iterrows():  # go through one contract at a time
        # Step 1: collect every clause that is actually written in THIS contract,
        # as {category name: clause text}. These real texts are the only ones we
        # are allowed to borrow from when we need a "No" example for this contract.
        present = {c: nonempty_context(row, c)
                   for c in task1_categories if nonempty_context(row, c)}
        # Step 2: ask the yes/no question once for every one of the 32 categories.
        for category in task1_categories:
            if to_binary(row[f"{category}_Answer"]) == "Yes":
                # Step 3a (YES case): the contract really has this clause, so use
                # the category's own real text as the input and label it "Yes".
                # Guard: the _Answer column can say "Yes" while the matching
                # context cell is blank (the two columns disagree). Without usable
                # text there is nothing to train on, so skip instead of KeyError.
                text = present.get(category)
                if not text:
                    continue
                label = "Yes"
            else:
                # Step 3b (NO case = hard negative): the contract does NOT have
                # this clause. Instead of a placeholder, gather the real clauses
                # from all the OTHER categories present in this contract...
                others = [t for c, t in present.items() if c != category]
                if not others:
                    continue  # ...if there is nothing else to borrow, skip this one
                # ...and pick one of those real clauses at random, labelled "No".
                text, label = random.choice(others), "No"
            # Step 4: save the finished example: the question, the text, the answer.
            rows.append({
                "instruction": f'Is the following contract text a "{category}" clause? Answer strictly "Yes" or "No".',
                "category": category,
                "input": text,
                "output": label,
            })
    return rows

# Step 4 (pipeline): split by CONTRACT first, then build examples from each side,
# so no single contract's clauses end up in both train and validation.
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
train_data = build_examples(train_df)
val_data = build_examples(val_df)

print(f"Contracts — train: {len(train_df)}, val: {len(val_df)}")
print(f"Examples  — train: {len(train_data)}, val: {len(val_data)}")

In [ ]:
for train, val in zip(train_data[:3], val_data[:3]):
    print("TRAIN EXAMPLE:")
    print(json.dumps(train, indent=2))
    print("\nVAL EXAMPLE:")
    print(json.dumps(val, indent=2))
    print("\n" + "="*50 + "\n")

# Step 5 : Balance classes on the **train** split only

Hard negatives can still be a minority/majority depending on how many clauses each contract has, and without balancing the model drifts to always answering the majority class. Downsample the majority class toward ~1:1 on **train only**; leave `val_data` at its natural distribution so validation metrics stay honest.

In [ ]:
from collections import defaultdict

def balance(data, ratio=1.0, seed=42):
    rng = random.Random(seed)
    pos = [e for e in data if e["output"] == "Yes"]
    neg = [e for e in data if e["output"] == "No"]
    keep_neg = min(len(neg), int(len(pos) * ratio))
    neg = rng.sample(neg, keep_neg)
    out = pos + neg
    rng.shuffle(out)
    return out

train_data = balance(train_data, ratio=1.0)   # train only — val untouched
print(f"After balancing — train: {len(train_data)} examples")

# Step 6a : Sanity checks — class counts + verify hard negatives are real text

Two quick checks before saving:
1. Print the final `Yes`/`No` counts (train is balanced; val is natural).
2. **Inspect a few `No` examples** — their `input` must be *real clause text borrowed from another category*, never the old `[No matching clause excerpt found...]` placeholder. If a `No` input is a placeholder, the leak isn't closed.

The headline per-class precision / recall / F1 + confusion matrix is computed **after training** in Step 6b.

In [ ]:
from collections import Counter

def label_counts(data):
    return Counter(ex["output"] for ex in data)

print("Train label counts:", dict(label_counts(train_data)))
print("Val   label counts:", dict(label_counts(val_data)))

# Verify hard negatives: every `No` input must be REAL clause text, not a placeholder.
print("\nSample of `No` examples (inputs must be real borrowed clause text):")
no_examples = [ex for ex in train_data if ex["output"] == "No"]
for ex in no_examples[:3]:
    print(f"\n  category : {ex['category']}")
    print(f"  input    : {ex['input'][:200]}{'...' if len(ex['input']) > 200 else ''}")

assert all("[No matching clause excerpt found" not in ex["input"] for ex in no_examples), \
    "Found placeholder text in a No example — the leak is NOT closed."
print("\nOK — no placeholder strings found in `No` inputs.")

# Save examples to JSONL (ensure dirs exist; save paths == load paths)

In [ ]:
# All generated artifacts for this baseline go under OUTPUT_DIR (a dedicated
# subdirectory) so the fine-tuned run's files are never overwritten. The validation
# JSONL written here is identical to the fine-tuned run's (same seed / same pipeline).
CUAD_TRAIN_PATH = OUTPUT_DIR/'cuad'/'train'
CUAD_VALIDATION_PATH = OUTPUT_DIR/'cuad'/'validation'

# Ensure the train/ and validation/ directories exist before saving.
CUAD_TRAIN_PATH.mkdir(parents=True, exist_ok=True)
CUAD_VALIDATION_PATH.mkdir(parents=True, exist_ok=True)


In [ ]:
save_jsonl(train_data, CUAD_TRAIN_PATH/'cuad_train.jsonl')
save_jsonl(val_data, CUAD_VALIDATION_PATH/'cuad_validation.jsonl')
print(f"Saved {len(train_data)} training samples and {len(val_data)} validation samples.")

# Load the base model for inference (no training)

Instead of fine-tuning, load **`meta-llama/Meta-Llama-3.1-8B`** in 4-bit NF4 — exactly how the fine-tuning notebook loads its *base* model — then use it directly for generation. There is **no `SFTTrainer`, no `LoraConfig`, no completion-only loss, and no adapter saved to disk**. We only need the frozen base model to produce `Yes`/`No` predictions on the validation set built above.


- Note : before execution of the cell below run to the terminal `$env:HF_TOKEN=your_hf_token`

In [ ]:
# Diagnostic: confirm WHICH account the token belongs to and whether it can access the gated repo.
# A 403 "not in the authorized list" means the token is valid but this account lacks access.
import os
from huggingface_hub import whoami, auth_check
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

def get_hf_token():
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv("HF_TOKEN")

hf_token = get_hf_token()
assert hf_token, "HF_TOKEN not found (Kaggle Secret or local .env)"

# Base model for the real (non-smoke-test) run. 8B in 4-bit NF4 is ~5-6 GB of weights,
# which fits fully in a single T4 (16 GB) VRAM with no CPU/disk offload. Smoke-test was "meta-llama/Llama-3.2-1B".
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B"

# 1) Which account is this token? Request access on the model page with THIS exact account.
me = whoami(token=hf_token)
print(f"Token belongs to: {me['name']}  (type: {me.get('type')})")

# 2) Does that account actually have access to the gated repo?
try:
    auth_check(MODEL_ID, token=hf_token)
    print(f"✅ Access granted to {MODEL_ID} — you can run the load cell below.")
except GatedRepoError:
    print(f"❌ Still gated for account '{me['name']}'.")
    print(f"   -> Visit https://huggingface.co/{MODEL_ID} while logged in as '{me['name']}', "
          f"accept the license, and wait for approval.")
    print(f"   -> Or use the ungated mirror: model_name = 'unsloth/Meta-Llama-3.1-8B'")
except HfHubHTTPError as e:
    print(f"❌ Auth/HTTP error (likely an invalid or expired token): {e}")


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
import os

def get_hf_token():
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    from dotenv import load_dotenv
    load_dotenv()  # reads .env from the current working dir (project root)
    return os.getenv("HF_TOKEN")

hf_token = get_hf_token()
assert hf_token, "HF_TOKEN not found (Kaggle Secret or local .env)"

from huggingface_hub import login
login(token=hf_token)

# 1. Configuration - base model, loaded AS-IS (NO fine-tuning, NO adapter).
# 8B in 4-bit NF4 is ~5-6 GB of weights, which fits fully in a single T4 (16 GB).
model_name = "meta-llama/Meta-Llama-3.1-8B"
MAX_SEQ_LENGTH = 1024   # single clauses are short; matches the fine-tuned run

# 2. QLoRA-style 4-bit load (same quantization the fine-tuned run used for its base model)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 3. Load Base Model (whole model on GPU 0; GPU 1 hidden in cell 1)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=hf_token
)

# 4. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loaded base model '{model_name}' in 4-bit for inference (no training).")


# Evaluate the base model on validation — per-class precision / recall / F1

Generate a `Yes`/`No` prediction for every validation example **using the un-fine-tuned base model** and report per-class precision / recall / F1 + a confusion matrix with `sklearn.metrics.classification_report`. Under class imbalance, accuracy and loss are meaningless ("always No" can score 80%+), so we look at the per-class breakdown.

These are the **baseline** numbers: comparing them against the fine-tuned notebook's metrics on this identical validation set shows exactly what fine-tuning bought. Results are written to the **`no_finetune_baseline/` subdirectory** (`eval_metrics.json`, `eval_report.txt`) so they sit alongside — but never overwrite — the fine-tuned run's artifacts.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Use cache for faster generation at inference time.
model.config.use_cache = True
model.eval()

def predict(example):
    prompt = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=1024).to(model.device)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        out = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return "Yes" if "yes" in decoded.strip().lower() else "No"

y_true = [ex["output"] for ex in val_data]
y_pred = [predict(ex) for ex in val_data]

labels = ["Yes", "No"]
report_str = classification_report(y_true, y_pred, labels=labels, zero_division=0)
report_dict = classification_report(y_true, y_pred, labels=labels, zero_division=0,
                                    output_dict=True)
cm = confusion_matrix(y_true, y_pred, labels=labels)

print("Per-class precision / recall / F1 on validation (BASELINE, no fine-tuning):\n")
print(report_str)
print("Confusion matrix (rows = true [Yes, No], cols = pred [Yes, No]):")
print(cm)

# --- Persist evaluation results as downloadable artifacts ---
# Written under OUTPUT_DIR (no_finetune_baseline/) so `kaggle kernels output` pulls
# them back into a subfolder that never overwrites the fine-tuned run's metrics.
import json

eval_metrics = {
    "model_name": model_name,
    "fine_tuned": False,
    "n_validation_examples": len(y_true),
    "accuracy": report_dict["accuracy"],
    "classification_report": report_dict,
    "confusion_matrix": {
        "labels": labels,
        "rows_are_true_cols_are_pred": True,
        "matrix": cm.tolist(),
    },
}
eval_metrics_path = OUTPUT_DIR / "eval_metrics.json"
with open(eval_metrics_path, "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, indent=2)

eval_report_path = OUTPUT_DIR / "eval_report.txt"
with open(eval_report_path, "w", encoding="utf-8") as f:
    f.write("Per-class precision / recall / F1 on validation (BASELINE, no fine-tuning):\n\n")
    f.write(report_str + "\n\n")
    f.write("Confusion matrix (rows = true [Yes, No], cols = pred [Yes, No]):\n")
    f.write(str(cm) + "\n")

print(f"\nWrote eval metrics to {eval_metrics_path}")
print(f"Wrote eval report  to {eval_report_path}")
